In [1]:
%pip install -q coqui-tts coqui-tts-trainer

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["COQUI_TOS_AGREED"] = "1"

import pandas as pd, numpy as np, soundfile as sf, librosa
from pathlib import Path
from tqdm.auto import tqdm

DATA_ROOT = Path("/kaggle/input/datasets/alaaldeais/najdi-dataset")
OUT_DIR   = Path("/kaggle/working/xtts_data")
WAV_OUT   = OUT_DIR / "wavs"
WAV_OUT.mkdir(parents=True, exist_ok=True)
TARGET_SR, SPEAKER = 22050, "najdi_speaker"
TEST_SESSIONS, EVAL_SESSIONS = [14], [13]

df = pd.read_csv(DATA_ROOT / "metadata.csv")
on_disk = {p.name for p in (DATA_ROOT / "wavs").glob("*.wav")}
df["file_name"] = df["audio_path"].astype(str).apply(lambda p: Path(p).name)
df = df[df["file_name"].isin(on_disk)].reset_index(drop=True)
print("Matched clips:", len(df))

rows = []
for _, r in tqdm(df.iterrows(), total=len(df), desc="Converting"):
    out_name = f"clip_{int(r['id']):05d}.wav"
    out_path = WAV_OUT / out_name
    if not out_path.exists():
        wav, sr = sf.read(DATA_ROOT / "wavs" / r["file_name"], dtype="float32")
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        wav = librosa.resample(wav, orig_sr=sr, target_sr=TARGET_SR)
        _, (s, e) = librosa.effects.trim(wav, top_db=40)
        pad = int(0.15 * TARGET_SR)
        wav = wav[max(0, s - pad):min(len(wav), e + pad)]
        sf.write(out_path, wav, TARGET_SR, subtype="PCM_16")
    rows.append({"audio_file": f"wavs/{out_name}", "text": str(r["Text"]).strip(),
                 "speaker_name": SPEAKER, "recorded_at": r["recorded_at"],
                 "dur": sf.info(out_path).duration})

meta = pd.DataFrame(rows)
meta["recorded_at"] = pd.to_datetime(meta["recorded_at"])
meta = meta.sort_values("recorded_at").reset_index(drop=True)
meta["session"] = (meta["recorded_at"].diff() > pd.Timedelta(minutes=30)).cumsum()

meta["split"] = "train"
meta.loc[meta["session"].isin(EVAL_SESSIONS), "split"] = "eval"
meta.loc[meta["session"].isin(TEST_SESSIONS), "split"] = "test"
for split in ["train", "eval", "test"]:
    part = meta[meta["split"] == split]
    part[["audio_file", "text", "speaker_name"]].to_csv(OUT_DIR / f"metadata_{split}.csv", sep="|", index=False)
    print(f"{split:5s}: {len(part):3d} clips | {part['dur'].sum()/60:5.1f} min")
meta.to_csv(OUT_DIR / "metadata_full_with_sessions.csv", index=False)
print("Data ready ✅")

Matched clips: 756


Converting:   0%|          | 0/756 [00:00<?, ?it/s]

train: 701 clips |  92.6 min
eval :  35 clips |   6.2 min
test :  20 clips |   3.8 min
Data ready ✅


In [3]:
%%writefile /kaggle/working/train_xtts_najdi.py
import os
os.environ["COQUI_TOS_AGREED"] = "1"

from huggingface_hub import hf_hub_download
from trainer import Trainer, TrainerArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig
from TTS.tts.models.xtts import XttsAudioConfig

DATA_DIR = "/kaggle/working/xtts_data"
OUT_PATH = "/kaggle/working/xtts_runs"
BASE_DIR = "/kaggle/working/xtts_base"

EPOCHS      = 8
BATCH_SIZE  = 2
GRAD_ACCUM  = 1
LR          = 5e-6
MAX_WAV_SEC = 19

files = {f: hf_hub_download("coqui/XTTS-v2", f, local_dir=BASE_DIR)
         for f in ["model.pth", "dvae.pth", "mel_stats.pth", "vocab.json", "config.json"]}

dataset = BaseDatasetConfig(
    formatter="coqui", dataset_name="najdi", path=DATA_DIR,
    meta_file_train="metadata_train.csv", meta_file_val="metadata_eval.csv", language="ar",
)

def main():
    model_args = GPTArgs(
        max_conditioning_length=132300,
        min_conditioning_length=66150,
        debug_loading_failures=False,
        max_wav_length=int(MAX_WAV_SEC * 22050),
        max_text_length=200,
        mel_norm_file=files["mel_stats.pth"],
        dvae_checkpoint=files["dvae.pth"],
        xtts_checkpoint=files["model.pth"],
        tokenizer_file=files["vocab.json"],
        gpt_num_audio_tokens=1026,
        gpt_start_audio_token=1024,
        gpt_stop_audio_token=1025,
        gpt_use_masking_gt_prompt_approach=True,
        gpt_use_perceiver_resampler=True,
    )
    audio = XttsAudioConfig(sample_rate=22050, dvae_sample_rate=22050, output_sample_rate=24000)

    config = GPTTrainerConfig(
        datasets=[dataset],
        epochs=EPOCHS, output_path=OUT_PATH, model_args=model_args,
        run_name="xtts_najdi", project_name="najdi_tts",
        run_description="XTTS-v2 GPT fine-tune on own Najdi recordings",
        dashboard_logger="tensorboard", audio=audio,
        batch_size=BATCH_SIZE, eval_batch_size=BATCH_SIZE, batch_group_size=48,
        num_loader_workers=2, eval_split_max_size=256,
        print_step=25, plot_step=100, log_model_step=1000,
        save_step=1_000_000,
        save_n_checkpoints=1, save_checkpoints=True,
        save_best_after=0,
        print_eval=False,
        optimizer="AdamW", optimizer_wd_only_on_weights=True,
        optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
        lr=LR, lr_scheduler="MultiStepLR",
        lr_scheduler_params={"milestones": [50000 * 18, 150000 * 18, 300000 * 18],
                             "gamma": 0.5, "last_epoch": -1},
        test_sentences=[],
    )

    model = GPTTrainer(config)
    train_samples, eval_samples = load_tts_samples(
        config, eval_split=True,
        eval_split_max_size=config.eval_split_max_size,
        eval_split_size=config.eval_split_size,
    )
    print(f"Train samples: {len(train_samples)} | Eval samples: {len(eval_samples)}")

    trainer = Trainer(
        TrainerArgs(restore_path=None, skip_train_epoch=False,
                    start_with_eval=True, grad_accum_steps=GRAD_ACCUM),
        config, output_path=OUT_PATH, model=model,
        train_samples=train_samples, eval_samples=eval_samples,
    )
    trainer.fit()

if __name__ == "__main__":
    main()

Writing /kaggle/working/train_xtts_najdi.py


In [4]:
!CUDA_VISIBLE_DEVICES=0 python -u /kaggle/working/train_xtts_najdi.py 2>&1 | tee /kaggle/working/train_log.txt

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
Train samples: 701 | Eval samples: 35
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: False
 | > Precision: float32
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 4
 | > Num. of Torch Threads: 1
 | > Torch seed: 1
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
I0000 

In [5]:
import torch, shutil, glob, re
from pathlib import Path

run_dir = sorted(glob.glob("/kaggle/working/xtts_runs/xtts_najdi-*"))[-1]
best = max(glob.glob(f"{run_dir}/best_model_*.pth"),
           key=lambda p: int(re.search(r"best_model_(\d+)", p).group(1)))
print("Best checkpoint:", best)

FINAL = Path("/kaggle/working/najdi_xtts_final")
FINAL.mkdir(exist_ok=True)

ckpt = torch.load(best, map_location="cpu", weights_only=False)
torch.save({"model": ckpt["model"]}, FINAL / "model.pth")
for f in ["config.json", "vocab.json"]:
    shutil.copy(f"/kaggle/working/xtts_base/{f}", FINAL / f)
shutil.copy("/kaggle/working/train_log.txt", FINAL / "train_log.txt")
shutil.copy("/kaggle/working/xtts_data/metadata_full_with_sessions.csv", FINAL)

shutil.rmtree("/kaggle/working/xtts_runs")
shutil.rmtree("/kaggle/working/xtts_base")
shutil.rmtree("/kaggle/working/xtts_data/wavs")

for p in FINAL.iterdir():
    print(f"{p.name:40s} {p.stat().st_size/1e9:.2f} GB")

Best checkpoint: /kaggle/working/xtts_runs/xtts_najdi-November-14-2026_05+05PM-0000000/best_model_2106.pth
vocab.json                               0.00 GB
model.pth                                2.08 GB
metadata_full_with_sessions.csv          0.00 GB
train_log.txt                            0.00 GB
config.json                              0.00 GB
